# PJM LMPs Hourly

In [6]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add backend/src to path so we can import the utils
REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "backend"))

from src.utils.azure_postgresql import pull_from_db

In [7]:
# Read SQL query from file
sql_path = Path.cwd() / "pjm_lmps_hourly.sql"
query = sql_path.read_text()

# Pull data
df = pull_from_db(query=query)
print(f"{len(df):,} rows")
df.head()

5,712 rows


,datetime,date,hour_ending,hub,market,lmp_total,lmp_system_energy_price,lmp_congestion_price,lmp_marginal_loss_price
0,2026-03-04 01:00:00,2026-03-04,1.0,AEP-DAYTON HUB,rt,81.185833,80.646666,1.962500,-1.423333
1,2026-03-04 01:00:00,2026-03-04,1.0,AEP-DAYTON HUB,da,36.141938,36.330000,1.305087,-1.493149
2,2026-03-04 01:00:00,2026-03-04,1.0,AEP-DAYTON HUB,dart,-45.043895,-44.316666,-0.657413,-0.069816
3,2026-03-04 01:00:00,2026-03-04,1.0,AEP GEN HUB,da,35.458596,36.330000,1.062530,-1.933934
4,2026-03-04 01:00:00,2026-03-04,1.0,AEP GEN HUB,rt,79.416667,80.646667,1.685000,-2.915000


In [8]:
df.dtypes

datetime                   datetime64[us]
date                               object
hour_ending                       float64
hub                                   str
market                                str
lmp_total                         float64
lmp_system_energy_price           float64
lmp_congestion_price              float64
lmp_marginal_loss_price           float64
dtype: object

## Total LMP by Hub and Market

In [9]:
for market in df["market"].unique():
    df_market = df[df["market"] == market]
    fig = px.line(
        df_market,
        x="datetime",
        y="lmp_total",
        color="hub",
        title=f"Total LMP - {market}",
        labels={"lmp_total": "LMP Total ($/MWh)", "datetime": "Date/Hour"},
    )
    fig.update_layout(template="plotly_dark", height=500)
    fig.show()

## LMP Components (Energy, Congestion, Loss) by Hub

In [10]:
components = ["lmp_system_energy_price", "lmp_congestion_price", "lmp_marginal_loss_price"]
component_labels = ["System Energy", "Congestion", "Marginal Loss"]

for hub in df["hub"].unique():
    df_hub = df[df["hub"] == hub]

    fig = make_subplots(
        rows=len(components), cols=1,
        shared_xaxes=True,
        subplot_titles=component_labels,
        vertical_spacing=0.06,
    )

    for i, (comp, label) in enumerate(zip(components, component_labels), start=1):
        for market in df_hub["market"].unique():
            df_slice = df_hub[df_hub["market"] == market]
            fig.add_trace(
                go.Scatter(
                    x=df_slice["datetime"],
                    y=df_slice[comp],
                    name=f"{market}",
                    legendgroup=market,
                    showlegend=(i == 1),
                ),
                row=i, col=1,
            )
        fig.update_yaxes(title_text="$/MWh", row=i, col=1)

    fig.update_layout(
        title=f"LMP Components - {hub}",
        template="plotly_dark",
        height=300 * len(components),
    )
    fig.show()

## DA vs RT Spread

In [11]:
# Pivot to get DA and RT side-by-side, then compute spread
if set(["DA", "RT"]).issubset(df["market"].unique()):
    df_pivot = df.pivot_table(
        index=["date", "hour_ending", "hub", "datetime"],
        columns="market",
        values="lmp_total",
    ).reset_index()
    df_pivot["DA_RT_Spread"] = df_pivot["DA"] - df_pivot["RT"]

    fig = px.bar(
        df_pivot,
        x="datetime",
        y="DA_RT_Spread",
        color="hub",
        title="DA - RT LMP Spread",
        labels={"DA_RT_Spread": "Spread ($/MWh)", "datetime": "Date/Hour"},
        barmode="group",
    )
    fig.update_layout(template="plotly_dark", height=500)
    fig.show()
else:
    print("Both DA and RT markets not present in data.")

Both DA and RT markets not present in data.
